# New-RscMssqlSnapshot
DBAs will often take a backup before they make a change to the database. In general, you need 2 things to take a snapshot of a database, the database ID and the SLA ID. An SLA is required for all on demand snapshots to manage the retention of the snapshot. 

# Workflow
To start an On-Demand Snapsot of a database, you need to do the following
1. Use **Connect-RSC** to Connect to RSC
1. Use **Get-RscCluster** to get the cluster id that the SQL Server is registered. This is akin to the datacenter that the SQL Server is located in. While you can run the next command without the cluster ID, you may get more records back than expected. This is because your SQL Server Host could exist on 2 different cluster. Once as a replicated source and once as the replicated target. 
1. Use **Get-RscMssqlInstance** to get the instance id of the SQL Server based on the name of the instance and the cluster where it is located. 
1. Use **Get-RscMssqlDatabase** to get the database ID based on the database name and the SQL Server Instance. 
1. Use **Get-RscSLADomain** to get the ID SLA Domain that will manage the retention of the snapshot.
1. Use **New-RscMssqlSnapshot** to initiate an On Demand Snapshot of a database. 
1. Use **Get-RscRequest** to track the request and know when it completes. ***DOES NOT EXIST YET***
1. Use **Disconnect-RSC** to disconnect from RSC

In [2]:
$ClusterName = "cl-rbk-edge-1"
$SourceHostName = "cl-mssql-1.tpm.lab"
$DatabaseName = "AdventureWorks2022"
$TargetHostName = "cl-mssql-1.tpm.lab"
$SLADomain = "CL-MSSQL"
$AvailabilityGroupName = "AG1"

# Connect to RSC

In [3]:
Import-Module RubrikSecurityCloud
$ConnectRsc = @{
    Server = op read "op://Rubrik/TPMServiceAccount/hostname"
    ClientId = op read "op://Rubrik/TPMServiceAccount/username"
    ClientSecret = ConvertTo-SecureString (op read "op://Rubrik/TPMServiceAccount/credential") -AsPlainText -Force
}
Connect-RSC @ConnectRsc

ServiceAccountDetails


ClientId,client|c6ab21c8-9933-488a-8b39-6d24ea8d2037
AuthenticationState,AUTHORIZED
SdkSchemaVersion,v20250616-31
ServerVersion,v20250714-24


# Get the Cluster ID

In [4]:
$RscCluster = Get-RscCluster -Name $ClusterName
$RscCluster

key,type,value
PauseStatus,RubrikSecurityCloud.Types.ClusterPauseStatus,NOT_PAUSED
Status,RubrikSecurityCloud.Types.ClusterStatus,CONNECTED
StatusFromDb,RubrikSecurityCloud.Types.ClusterConnectionStatusFromDb,CONNECTED
SubStatus,RubrikSecurityCloud.Types.ClusterSubStatus,DEFAULT
SystemStatus,RubrikSecurityCloud.Types.ClusterSystemStatus,OK
Type,RubrikSecurityCloud.Types.ClusterTypeEnum,ROBO
Id,System.String,10355ecf-8233-45bb-88d2-5db926d96b4c
IsAirGapped,System.Boolean,False
IsClusterRemovalTprEnabled,System.Boolean,False
Name,System.String,cl-rbk-edge-1


# Get the Database based on the Instance

## Get the Instance Information

In [4]:
$RscMssqlInstance = Get-RscMssqlInstance -RscCluster $RscCluster -HostName $SourceHostName
$RscMssqlInstance

RubrikSecurityCloud.Types.MssqlInstance AuthorizedOperations <null> CdmPendingObjectPauseAssignment <null> ObjectType <null> SlaAssignment <null> ConfiguredSlaDomain <null> EffectiveRetentionSlaDomain <null> EffectiveSlaDomain RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes <null> RetentionLockMode <null> ClusterUuid <null> Description <null> Id 99e55090-f8ab-4f5e-8956-ccf4bbfae442 IsArchived <null> IsDefault <null> IsReadOnly <null> IsRetentionLockedSla <null> Name CL-MSSQL OwnerOrgName <null> ProtectedObjectCount <null> SnapshotScheduleLastUpdatedAt <null> StateVersion <null> UiColor <null> Version <null> AllOrgsHavingAccess <null> AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo <null> ArchivalSpec <null> ArchivalSpecs <null> BackupWindows <null> BaseFrequency <null> ClusterToSyncStatusMap <null> FirstFullBackupWindows <null> LocalRetentionLimit <null> LogConfig <null> ObjectSpecificConfigs <null> OwnerOrg <null> PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 <null> SnapshotSchedule <null> SourceClusters <null> UpgradeInfo <null> PendingSla <null> ReplicatedObjects <null> HasLogConfigFromSla <null> HostLogRetention <null> Id def948c2-03c4-5172-86d9-97a0fa669c58 LogBackupFrequencyInSeconds <null> LogBackupRetentionInHours <null> Name MSSQLSERVER NumWorkloadDescendants <null> ReplicatedObjectCount <null> SlaPauseStatus <null> UnprotectableReasons <null> AllOrgs <null> AllTags <null> Cluster RubrikSecurityCloud.Types.Cluster CyberEventLockdownMode <null> EosStatus <null> LicensedProducts <null> PauseStatus <null> ProductType <null> RegisteredMode <null> Status <null> StatusFromDb <null> SubStatus <null> SystemStatus <null> Type <null> CdmRbacMigrationStatus <null> ConnectivityLastUpdated <null> DefaultAddress <null> DefaultPort <null> EncryptionEnabled <null> EosDate <null> EstimatedRunway <null> Id 10355ecf-8233-45bb-88d2-5db926d96b4c IsAirGapped <null> IsClusterRemovalTprEnabled <null> IsHealthy <null> IsTprEnabled <null> LastConnectionTime <null> Name cl-rbk-edge-1 NoSqlWorkloadCount <null> PassesConnectivityCheck <null> RawAddress <null> RegistrationTime <null> SnapshotCount <null> SystemStatusMessage <null> Timezone <null> Version <null> ActivitySeriesConnection <null> AllOrgs <null> AuthorizedOperations <null> CcprovisionInfo <null> CdmClusterNodeDetails <null> CdmNotificationSettings <null> CdmUpgradeInfo <null> CloudInfo <null> ClusterDiskConnection <null> ClusterNodeConnection <null> ClusterNodeStats <null> ConfigProtectionInfo <null> CyberEventLockdownSupportCaseDetails <null> DatagovAutoEnablePolicyConfig <null> DatagovPreviewerConfig <null> GeoLocation <null> GlobalManagerConnectivityStatus <null> IpmiInfo <null> LambdaConfig <null> LambdaFeatureHistory <null> MetadataPullScheduler <null> Metric <null> MetricTimeSeries <null> MetricTimeSeriesNew <null> ReplicationSources <null> ReplicationTargets <null> RubrikSyncStatus <null> SnappableConnection <null> State <null> SystemStatusAffectedNodes <null> WebServerCertificate <null> Vars RubrikSecurityCloud.Types.Cluster+InlineVars ActivitySeriesConnection key value first <null> after <null> last <null> before <null> sortBy <null> filters <null> ClusterDiskConnection key value first <null> after <null> last <null> before <null> filter <null> ClusterNodeConnection key value first <null> after <null> last <null> before <null> filter <null> sortBy <null> sortOrder <null> ClusterNodeStats key value cdmClusterNodeID <null> timeRange <null> MetricTimeSeries key value timeRange <null> unit <null> MetricTimeSeriesNew key value timeRange <null> unit <null> SnappableConnection key value first <null> after <null> last <null> before <null> filter <null> CrossAccountReplicatedObjectInfos <null> DescendantConnection <null> EffectiveSlaSourceObject <null> LatestUserNote <null> LogicalChildConnection <null> LogicalPath <null> ObjectPauseStatus <null> PendingObjectDeletionStatus <null> PhysicalPath <null> PrimaryClusterLocation <null> SecurityMetadata <null>

## Get the Database Information

In [5]:
$RscMssqlDatabase = Get-RscMssqlDatabase -Name $DatabaseName -RscMssqlInstance $RscMssqlInstance -Replica:$false -Relic:$false
$RscMssqlDatabase

RubrikSecurityCloud.Types.MssqlDatabase AuthorizedOperations <null> CdmPendingObjectPauseAssignment <null> ObjectType MSSQL SlaAssignment UNASSIGNED ConfiguredSlaDomain <null> EffectiveRetentionSlaDomain <null> EffectiveSlaDomain RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes <null> RetentionLockMode <null> ClusterUuid Description Id UNPROTECTED IsArchived False IsDefault False IsReadOnly False IsRetentionLockedSla False Name UNPROTECTED OwnerOrgName <null> ProtectedObjectCount <null> SnapshotScheduleLastUpdatedAt <null> StateVersion 0 UiColor <null> Version <null> AllOrgsHavingAccess <null> AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo <null> ArchivalSpec <null> ArchivalSpecs <null> BackupWindows <null> BaseFrequency <null> ClusterToSyncStatusMap <null> FirstFullBackupWindows <null> LocalRetentionLimit <null> LogConfig <null> ObjectSpecificConfigs <null> OwnerOrg <null> PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 <null> SnapshotSchedule <null> SourceClusters <null> UpgradeInfo <null> PendingSla <null> ReplicatedObjects <null> CdmId 8e334403-82ad-4f2a-b8ad-c94a97c55524 CdmLink <null> CdmOnDemandSnapshotCount <null> CopyOnly False DagId d240c948-bff9-5dd4-95d1-5497fa22c3a2 HasLogConfigFromSla True HasPermissions True HostLogRetention -1 Id 8d761fa8-f2c1-5891-9ef4-284c56052c9f IsInAvailabilityGroup False IsLogShippingSecondary False IsMount False IsOnline True IsRelic False LogBackupFrequencyInSeconds -2 LogBackupRetentionInHours -3 Name AdventureWorks2022s NumWorkloadDescendants 0 OnDemandSnapshotCount <null> PostBackupScript PreBackupScript RecoveryModel <null> ReplicatedObjectCount <null> SlaPauseStatus False UnprotectableReasons <null> Version 16.0.1140.6 AllOrgs <null> AllTags <null> CdmGroupedSnapshots <null> CdmNewestSnapshot <null> CdmOldestSnapshot <null> CdmSnapshots <null> Cluster RubrikSecurityCloud.Types.Cluster CyberEventLockdownMode <null> EosStatus <null> LicensedProducts <null> PauseStatus <null> ProductType <null> RegisteredMode <null> Status <null> StatusFromDb <null> SubStatus <null> SystemStatus <null> Type <null> CdmRbacMigrationStatus <null> ConnectivityLastUpdated <null> DefaultAddress <null> DefaultPort <null> EncryptionEnabled <null> EosDate <null> EstimatedRunway <null> Id 10355ecf-8233-45bb-88d2-5db926d96b4c IsAirGapped <null> IsClusterRemovalTprEnabled <null> IsHealthy <null> IsTprEnabled <null> LastConnectionTime <null> Name cl-rbk-edge-1 NoSqlWorkloadCount <null> PassesConnectivityCheck <null> RawAddress <null> RegistrationTime <null> SnapshotCount <null> SystemStatusMessage <null> Timezone <null> Version <null> ActivitySeriesConnection <null> AllOrgs <null> AuthorizedOperations <null> CcprovisionInfo <null> CdmClusterNodeDetails <null> CdmNotificationSettings <null> CdmUpgradeInfo <null> CloudInfo <null> ClusterDiskConnection <null> ClusterNodeConnection <null> ClusterNodeStats <null> ConfigProtectionInfo <null> CyberEventLockdownSupportCaseDetails <null> DatagovAutoEnablePolicyConfig <null> DatagovPreviewerConfig <null> GeoLocation <null> GlobalManagerConnectivityStatus <null> IpmiInfo <null> LambdaConfig <null> LambdaFeatureHistory <null> MetadataPullScheduler <null> Metric <null> MetricTimeSeries <null> MetricTimeSeriesNew <null> ReplicationSources <null> ReplicationTargets <null> RubrikSyncStatus <null> SnappableConnection <null> State <null> SystemStatusAffectedNodes <null> WebServerCertificate <null> Vars RubrikSecurityCloud.Types.Cluster+InlineVars ActivitySeriesConnection key value first <null> after <null> last <null> before <null> sortBy <null> filters <null> ClusterDiskConnection key value first <null> after <null> last <null> before <null> filter <null> ClusterNodeConnection key value first <null> after <null> last <null> before <null> filter <null> sortBy <null> sortOrder <null> ClusterNodeStats key value cdmClusterNodeID <null> timeRange <null> MetricTimeSeries key value timeRange <null> unit <null> MetricTimeSeriesNew key value timeRange <null> 

# Get the Database based on the AG

## Get the Availability Group 

In [5]:
$RscMssqlAvailabilityGroup = Get-RscMssqlAvailabilityGroup -AvailabilityGroupName $AvailabilityGroupName -RscCluster $RscCluster -Replica:$false -Relic:$false
$RscMssqlAvailabilityGroup

RubrikSecurityCloud.Types.MssqlAvailabilityGroup AuthorizedOperations <null> CdmPendingObjectPauseAssignment <null> ObjectType <null> SlaAssignment <null> ConfiguredSlaDomain <null> EffectiveRetentionSlaDomain <null> EffectiveSlaDomain RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes <null> RetentionLockMode <null> ClusterUuid <null> Description <null> Id 99e55090-f8ab-4f5e-8956-ccf4bbfae442 IsArchived <null> IsDefault <null> IsReadOnly <null> IsRetentionLockedSla <null> Name CL-MSSQL OwnerOrgName <null> ProtectedObjectCount <null> SnapshotScheduleLastUpdatedAt <null> StateVersion <null> UiColor <null> Version <null> AllOrgsHavingAccess <null> AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo <null> ArchivalSpec <null> ArchivalSpecs <null> BackupWindows <null> BaseFrequency <null> ClusterToSyncStatusMap <null> FirstFullBackupWindows <null> LocalRetentionLimit <null> LogConfig <null> ObjectSpecificConfigs <null> OwnerOrg <null> PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 <null> SnapshotSchedule <null> SourceClusters <null> UpgradeInfo <null> PendingSla <null> ReplicatedObjects <null> CopyOnly <null> HasLogConfigFromSla <null> HostLogRetention <null> Id 92347e4d-5b18-5102-be9f-a33b7c1597c7 LogBackupFrequencyInSeconds <null> LogBackupRetentionInHours <null> Name AG1 NumWorkloadDescendants <null> ReplicatedObjectCount <null> SlaPauseStatus <null> AllOrgs <null> AllTags <null> Cluster RubrikSecurityCloud.Types.Cluster CyberEventLockdownMode <null> EosStatus <null> LicensedProducts <null> PauseStatus <null> ProductType <null> RegisteredMode <null> Status <null> StatusFromDb <null> SubStatus <null> SystemStatus <null> Type <null> CdmRbacMigrationStatus <null> ConnectivityLastUpdated <null> DefaultAddress <null> DefaultPort <null> EncryptionEnabled <null> EosDate <null> EstimatedRunway <null> Id 10355ecf-8233-45bb-88d2-5db926d96b4c IsAirGapped <null> IsClusterRemovalTprEnabled <null> IsHealthy <null> IsTprEnabled <null> LastConnectionTime <null> Name cl-rbk-edge-1 NoSqlWorkloadCount <null> PassesConnectivityCheck <null> RawAddress <null> RegistrationTime <null> SnapshotCount <null> SystemStatusMessage <null> Timezone <null> Version <null> ActivitySeriesConnection <null> AllOrgs <null> AuthorizedOperations <null> CcprovisionInfo <null> CdmClusterNodeDetails <null> CdmNotificationSettings <null> CdmUpgradeInfo <null> CloudInfo <null> ClusterDiskConnection <null> ClusterNodeConnection <null> ClusterNodeStats <null> ConfigProtectionInfo <null> CyberEventLockdownSupportCaseDetails <null> DatagovAutoEnablePolicyConfig <null> DatagovPreviewerConfig <null> GeoLocation <null> GlobalManagerConnectivityStatus <null> IpmiInfo <null> LambdaConfig <null> LambdaFeatureHistory <null> MetadataPullScheduler <null> Metric <null> MetricTimeSeries <null> MetricTimeSeriesNew <null> ReplicationSources <null> ReplicationTargets <null> RubrikSyncStatus <null> SnappableConnection <null> State <null> SystemStatusAffectedNodes <null> WebServerCertificate <null> Vars RubrikSecurityCloud.Types.Cluster+InlineVars ActivitySeriesConnection key value first <null> after <null> last <null> before <null> sortBy <null> filters <null> ClusterDiskConnection key value first <null> after <null> last <null> before <null> filter <null> ClusterNodeConnection key value first <null> after <null> last <null> before <null> filter <null> sortBy <null> sortOrder <null> ClusterNodeStats key value cdmClusterNodeID <null> timeRange <null> MetricTimeSeries key value timeRange <null> unit <null> MetricTimeSeriesNew key value timeRange <null> unit <null> SnappableConnection key value first <null> after <null> last <null> before <null> filter <null> CrossAccountReplicatedObjectInfos <null> DescendantConnection <null> EffectiveSlaSourceObject <null> Instances <null> LatestUserNote <null> LogicalChildConnection <null> LogicalPath <null> ObjectPauseStatus <null> PendingObjectDeletionStatus <null> PhysicalPath <null> PrimaryClusterLocation RubrikSecurityCloud.Type

### Get the Database Information

In [6]:
$RscMssqlDatabase = Get-RscMssqlDatabase -Name $DatabaseName -RscMssqlAvailabilityGroup $RscMssqlAvailabilityGroup
$RscMssqlDatabase

RubrikSecurityCloud.Types.MssqlDatabase AuthorizedOperations <null> CdmPendingObjectPauseAssignment <null> ObjectType MSSQL SlaAssignment DERIVED ConfiguredSlaDomain <null> EffectiveRetentionSlaDomain <null> EffectiveSlaDomain RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes <null> RetentionLockMode <null> ClusterUuid 00000000-0000-0000-0000-000000000000 Description Id 99e55090-f8ab-4f5e-8956-ccf4bbfae442 IsArchived False IsDefault False IsReadOnly False IsRetentionLockedSla False Name CL-MSSQL OwnerOrgName <null> ProtectedObjectCount <null> SnapshotScheduleLastUpdatedAt <null> StateVersion 3 UiColor <null> Version AllOrgsHavingAccess <null> AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo <null> ArchivalSpec <null> ArchivalSpecs <null> BackupWindows <null> BaseFrequency <null> ClusterToSyncStatusMap <null> FirstFullBackupWindows <null> LocalRetentionLimit <null> LogConfig <null> ObjectSpecificConfigs <null> OwnerOrg <null> PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 <null> SnapshotSchedule <null> SourceClusters <null> UpgradeInfo <null> PendingSla <null> ReplicatedObjects <null> CdmId 97b0d65b-8d10-4c6b-8bb8-1e6beb1f7d1c CdmLink <null> CdmOnDemandSnapshotCount <null> CopyOnly False DagId 3f2074b2-e886-5fa7-990d-83ccc1a9d093 HasLogConfigFromSla True HasPermissions True HostLogRetention -1 Id 2cb8ad8b-ecb6-51c6-b22c-37ef22da6085 IsInAvailabilityGroup True IsLogShippingSecondary False IsMount False IsOnline True IsRelic False LogBackupFrequencyInSeconds -2 LogBackupRetentionInHours -3 Name AdventureWorks2022 NumWorkloadDescendants 0 OnDemandSnapshotCount <null> PostBackupScript PreBackupScript RecoveryModel <null> ReplicatedObjectCount <null> SlaPauseStatus False UnprotectableReasons <null> Version 16.0.1140.6 AllOrgs <null> AllTags <null> CdmGroupedSnapshots <null> CdmNewestSnapshot <null> CdmOldestSnapshot <null> CdmSnapshots <null> Cluster RubrikSecurityCloud.Types.Cluster CyberEventLockdownMode <null> EosStatus <null> LicensedProducts <null> PauseStatus <null> ProductType <null> RegisteredMode <null> Status <null> StatusFromDb <null> SubStatus <null> SystemStatus <null> Type <null> CdmRbacMigrationStatus <null> ConnectivityLastUpdated <null> DefaultAddress <null> DefaultPort <null> EncryptionEnabled <null> EosDate <null> EstimatedRunway <null> Id 10355ecf-8233-45bb-88d2-5db926d96b4c IsAirGapped <null> IsClusterRemovalTprEnabled <null> IsHealthy <null> IsTprEnabled <null> LastConnectionTime <null> Name cl-rbk-edge-1 NoSqlWorkloadCount <null> PassesConnectivityCheck <null> RawAddress <null> RegistrationTime <null> SnapshotCount <null> SystemStatusMessage <null> Timezone <null> Version <null> ActivitySeriesConnection <null> AllOrgs <null> AuthorizedOperations <null> CcprovisionInfo <null> CdmClusterNodeDetails <null> CdmNotificationSettings <null> CdmUpgradeInfo <null> CloudInfo <null> ClusterDiskConnection <null> ClusterNodeConnection <null> ClusterNodeStats <null> ConfigProtectionInfo <null> CyberEventLockdownSupportCaseDetails <null> DatagovAutoEnablePolicyConfig <null> DatagovPreviewerConfig <null> GeoLocation <null> GlobalManagerConnectivityStatus <null> IpmiInfo <null> LambdaConfig <null> LambdaFeatureHistory <null> MetadataPullScheduler <null> Metric <null> MetricTimeSeries <null> MetricTimeSeriesNew <null> ReplicationSources <null> ReplicationTargets <null> RubrikSyncStatus <null> SnappableConnection <null> State <null> SystemStatusAffectedNodes <null> WebServerCertificate <null> Vars RubrikSecurityCloud.Types.Cluster+InlineVars ActivitySeriesConnection key value first <null> after <null> last <null> before <null> sortBy <null> filters <null> ClusterDiskConnection key value first <null> after <null> last <null> before <null> filter <null> ClusterNodeConnection key value first <null> after <null> last <null> before <null> filter <null> sortBy <null> sortOrder <null> ClusterNodeStats key value cdmClusterNodeID <null> timeRange <null> MetricTimeSeries key value timeRange <null> unit <null> 

# Get SLA Domain Information

In [7]:
$RscSlaDomain = Get-RscSla -Name $SLADomain
$RscSlaDomain

RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes index value 0 MSSQL_OBJECT_TYPE RetentionLockMode NO_MODE ClusterUuid 00000000-0000-0000-0000-000000000000 Description Id 99e55090-f8ab-4f5e-8956-ccf4bbfae442 IsArchived False IsDefault False IsReadOnly False IsRetentionLockedSla False Name CL-MSSQL OwnerOrgName <null> ProtectedObjectCount 87 SnapshotScheduleLastUpdatedAt 2025-04-17 17:57:13Z StateVersion 3 UiColor <null> Version AllOrgsHavingAccess (empty) AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo (empty) ArchivalSpec <null> ArchivalSpecs (empty) BackupWindows (empty) BaseFrequency RubrikSecurityCloud.Types.Duration Unit DAYS DurationField 1 ClusterToSyncStatusMap (empty) FirstFullBackupWindows (empty) LocalRetentionLimit RubrikSecurityCloud.Types.Duration Unit DAYS DurationField 7 LogConfig <null> ObjectSpecificConfigs RubrikSecurityCloud.Types.ObjectSpecificConfigs AwsNativeS3SlaConfig <null> AwsRdsConfig <null> AzureBlobConfig <null> AzureSqlDatabaseDbConfig <null> AzureSqlManagedInstanceDbConfig <null> Db2Config <null> InformixSlaConfig <null> ManagedVolumeSlaConfig <null> MongoConfig <null> MssqlConfig RubrikSecurityCloud.Types.MssqlConfig Frequency RubrikSecurityCloud.Types.Duration Unit MINUTES DurationField 5 LogRetention RubrikSecurityCloud.Types.Duration Unit MINUTES DurationField 10080 MysqldbSlaConfig <null> NcdSlaConfig <null> OracleConfig <null> PostgresDbClusterSlaConfig <null> SapHanaConfig <null> VmwareVmConfig <null> OwnerOrg RubrikSecurityCloud.Types.SlaAssociatedOrganization Id 137a3dbe-ce49-4be0-acb5-d5e14ed150cc Name global PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 index value 0 RubrikSecurityCloud.Types.ReplicationSpecV2 AwsRegion NOT_DEFINED AzureRegion NOT_DEFINED AwsTarget RubrikSecurityCloud.Types.AwsReplicationTarget Region NOT_DEFINED AccountId AccountName AzureTarget RubrikSecurityCloud.Types.AzureReplicationTarget Region NOT_DEFINED SubscriptionId SubscriptionName CascadingArchivalSpecs (empty) Cluster <null> ReplicationLocalRetentionDuration <null> ReplicationPairs <null> RetentionDuration RubrikSecurityCloud.Types.Duration Unit DAYS DurationField 5 TargetMapping <null> SnapshotSchedule RubrikSecurityCloud.Types.SnapshotSchedule Daily RubrikSecurityCloud.Types.DailySnapshotSchedule BasicSchedule RubrikSecurityCloud.Types.BasicSnapshotSchedule RetentionUnit DAYS Frequency 1 Retention 7 Hourly <null> Minute <null> Monthly <null> Quarterly <null> Weekly <null> Yearly <null> SourceClusters <null> UpgradeInfo <null>

# Initiate On Demand Snapshot

In [8]:
New-RscMssqlSnapshot -RscMssqlDatabase $RscMssqlDatabase -RscSlaDomain $RscSlaDomain

EndTime,<null>
Id,MSSQL_DB_BACKUP_06a247df-b0fb-413e-990c-9b9ace002bf1_d80138ff-a6dd-4d30-96e4-38a756054118:::0
NodeId,<null>
Progress,<null>
Result,<null>
StartTime,<null>
Status,QUEUED
Error,<null>
Links,<null>


## Track the status of the backup

---

# Disconnect from RSC
It is important to have each script you create end by disconnecting from RSC. If you do not, then you will run into issues with token authentication exhaustion. RSC allows a Service Account to have 10 concurrent open connections to RSC. Additional requests will be denied until existing connections are closed. 

You can easily disconnect your connection by running the below statement. 

In [ ]:
Disconnect-Rsc